# Fine-Tune Llama-3 on AML Compute Cluster — Orchestrated Pipeline

Two-job pipeline orchestrated from this notebook:

1. **CPU prep job** (only if needed) — downloads `ruslanmv/ai-medical-chatbot`,
   formats with the Llama-3 chat template, splits, writes to blob, and
   registers a versioned AML **Data asset**.
2. **GPU training job** — mounts the asset and runs QLoRA + LoRA + SFT on
   `Cluster-A100-1GPU`. No HF download, no tokenization at runtime.

The notebook itself does **only orchestration** — no dataset download locally.


## Architecture

```
┌───────────────────┐    ┌──────────────────────────┐    ┌──────────────────────────┐
│ this notebook     │    │ CPU job (prep_dataset)   │    │ GPU job (fine_tune)      │
│ ───────────────── │    │ ──────────────────────── │    │ ──────────────────────── │
│ try get(asset)    │    │ load_dataset(HF)         │    │ load_from_disk(mount)    │
│  ├ exists → skip  │ →  │ apply_chat_template      │ →  │ AutoModel + 4-bit NF4    │
│  └ missing →      │    │ train_test_split         │    │ LoRA + SFTTrainer        │
│    submit prep    │    │ save_to_disk(output)     │    │ save to ./outputs/       │
│ register Data     │    │ (no GPU needed)          │    │                          │
│ submit GPU job    │    └──────────────┬───────────┘    └──────────────▲───────────┘
│ stream + download │                   ▼                               │
└───────────────────┘    ┌──────────────────────────┐                   │
                         │ workspaceblobstore       │  Input(asset@ver) │
                         │ datasets/<asset>/<ver>/  │ ──────────────────┘
                         │ (registered as Data)     │
                         └──────────────────────────┘
```

### Prerequisites
- `00_aml_cc_prepare_sub_environment.ipynb` has been run for this IP (NSP,
  shared-key exemption, cluster MI + RBAC).
- `01_aml_cc_create_container_image.ipynb` has built the env
  `sft-finetune-cu13@latest`.
- `Cluster-A100-1GPU` exists with system-assigned managed identity.

### Tools
- [`transformers`](https://huggingface.co/docs/transformers/) +
  [`peft`](https://huggingface.co/docs/peft/) +
  [`trl`](https://huggingface.co/docs/trl/) +
  [`bitsandbytes`](https://huggingface.co/docs/bitsandbytes/) for QLoRA SFT.
- [`accelerate`](https://huggingface.co/docs/accelerate/) for multi-GPU.
- AML `MLClient` + `Data` asset + `command` + `Input`/`Output`.
- MLflow auto-logging via `report_to="mlflow"`.


## 1. Imports & AML client


In [6]:
import os
from azure.ai.ml import MLClient, Input, Output, command
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data, UserIdentityConfiguration
from dotenv import load_dotenv

config_file_path = os.path.join(".", "config", "germanywest.env")
load_dotenv(dotenv_path=config_file_path, override=True)

from utils.amlauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

ml_client = MLClient(
    credential, settings.subscription_id, settings.resource_group, settings.workspace,
)
print(f"Workspace : {ml_client.workspace_name}")
print(f"Region    : {ml_client.workspaces.get(settings.workspace).location}")

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Workspace : aml-ww-yw-dos
Region    : germanywestcentral


In [7]:
# ─── Environment / compute ─────────────────────────────────────────────────
ENV_NAME       = "sft-finetune-cu13"        # built by 01_aml_cc_create_container_image.ipynb
ENV_VERSION    = "latest"                    # or pin to a specific version

GPU_COMPUTE    = "Cluster-A100-1GPU"         # actual SFT job
# CPU prep: reusing the GPU cluster is simplest (zero extra setup — same MI,
# NSP, RBAC, env image already cached). For real cost savings, create a tiny
# CPU cluster (e.g. Standard_DS3_v2, 0–1 nodes) and switch this constant.
PREP_COMPUTE   = GPU_COMPUTE

# ─── Dataset asset (the orchestration target) ──────────────────────────────
DATA_ASSET     = "ai-medical-chatbot-llama3"
DATA_VERSION   = "1.0.0"                     # bump when any prep param changes

PREP_DATASET_NAME    = "ruslanmv/ai-medical-chatbot"
PREP_TOKENIZER_MODEL = "NousResearch/Meta-Llama-3-8B-Instruct"
PREP_NUM_ROWS        = 10000
PREP_EVAL_SIZE       = 0.1

# Deterministic blob location keeps the registered Data asset's `.path`
# stable across re-registrations and lets us re-run the prep job idempotently.
PREP_OUTPUT_URI = (
    f"azureml://datastores/workspaceblobstore/paths/"
    f"datasets/{DATA_ASSET}/v{DATA_VERSION}/"
)

# ─── Training hyperparams ──────────────────────────────────────────────────
BASE_MODEL       = "NousResearch/Meta-Llama-3-8B-Instruct"
FINETUNED_MODEL  = "llama3-8b-chat-doctor"
NUM_EPOCHS       = 1
MAX_STEPS        = 50          # -1 = full epochs; small int for a smoke test
BATCH_SIZE       = 1
GRAD_ACCUM       = 8
LEARNING_RATE    = 2e-4
LORA_R           = 8
LORA_ALPHA       = 32

print("Constants loaded.")
print(f"  Env        : {ENV_NAME}@{ENV_VERSION}")
print(f"  GPU compute: {GPU_COMPUTE}")
print(f"  Prep compute: {PREP_COMPUTE}")
print(f"  Data asset : {DATA_ASSET}@{DATA_VERSION}")
print(f"  Base model : {BASE_MODEL}")
print(f"  Epochs / max_steps : {NUM_EPOCHS} / {MAX_STEPS}")


Constants loaded.
  Env        : sft-finetune-cu13@latest
  GPU compute: Cluster-A100-1GPU
  Prep compute: Cluster-A100-1GPU
  Data asset : ai-medical-chatbot-llama3@1.0.0
  Base model : NousResearch/Meta-Llama-3-8B-Instruct
  Epochs / max_steps : 1 / 50


## 2. Ensure the dataset asset exists (CPU prep job, skipped if present)

Try to fetch the versioned Data asset. If it's already registered, skip
straight to the GPU training job. If not, submit the CPU prep job (which
writes formatted data to a deterministic blob path) and then register that
path as a versioned `Data` asset so the next run hits the fast path.


In [8]:
# Fast path: if the asset exists, skip the CPU prep job entirely.
# Slow path: submit a CPU prep job that writes formatted parquet shards to
# the deterministic blob URI, then register that URI as a versioned Data asset.
try:
    data_asset = ml_client.data.get(name=DATA_ASSET, version=DATA_VERSION)
    print(f"✅ Dataset asset already exists: {data_asset.name}@{data_asset.version}")
    print(f"   path: {data_asset.path}")
    print("   → skipping prep job; the training job below will mount this asset.")
except Exception as e:
    print(f"📥 Asset '{DATA_ASSET}@{DATA_VERSION}' not found — submitting prep job.")
    print(f"   ({type(e).__name__}: {e})\n")

    prep_job = command(
        code="./src",
        command=(
            "python prepare_dataset.py "
            f"--dataset_name={PREP_DATASET_NAME} "
            f"--tokenizer_model={PREP_TOKENIZER_MODEL} "
            f"--num_data_rows={PREP_NUM_ROWS} "
            f"--eval_size={PREP_EVAL_SIZE} "
            "--output_dir=${{outputs.data}}"
        ),
        environment=f"{ENV_NAME}@{ENV_VERSION}",
        compute=PREP_COMPUTE,
        identity=UserIdentityConfiguration(),
        outputs={
            "data": Output(
                type=AssetTypes.URI_FOLDER,
                path=PREP_OUTPUT_URI,
                mode="rw_mount",
            ),
        },
        experiment_name="dataset-prep",
        display_name=f"prep-{DATA_ASSET}-v{DATA_VERSION}",
        description=(
            f"Format {PREP_DATASET_NAME} ({PREP_NUM_ROWS} rows) with "
            f"{PREP_TOKENIZER_MODEL} chat template; write to blob for "
            f"asset {DATA_ASSET}@{DATA_VERSION}."
        ),
    )
    submitted_prep = ml_client.jobs.create_or_update(prep_job)
    print(f"Submitted prep job: {submitted_prep.name}")
    print(f"Studio: {submitted_prep.studio_url}\n")

    # Blocks until terminal; prints stdout/stderr as it streams.
    ml_client.jobs.stream(submitted_prep.name)

    final_prep = ml_client.jobs.get(submitted_prep.name)
    print(f"\nFinal prep status: {final_prep.status}")
    if final_prep.status != "Completed":
        raise RuntimeError(
            f"Prep job ended with status {final_prep.status} — see logs above."
        )

    # Register the deterministic blob path as a versioned Data asset.
    data_asset = ml_client.data.create_or_update(Data(
        name=DATA_ASSET,
        version=DATA_VERSION,
        type=AssetTypes.URI_FOLDER,
        path=PREP_OUTPUT_URI,
        description=(
            f"{PREP_DATASET_NAME} ({PREP_NUM_ROWS} rows) chat-formatted with "
            f"{PREP_TOKENIZER_MODEL}, split "
            f"{1 - PREP_EVAL_SIZE:.0%}/{PREP_EVAL_SIZE:.0%}, "
            "via datasets.save_to_disk."
        ),
        tags={
            "source_dataset": PREP_DATASET_NAME,
            "tokenizer":      PREP_TOKENIZER_MODEL,
            "num_rows":       str(PREP_NUM_ROWS),
            "source_job":     submitted_prep.name,
            "format":         "datasets.save_to_disk",
        },
    ))
    print(f"\n✅ Registered: {data_asset.name}@{data_asset.version}")
    print(f"   path: {data_asset.path}")


✅ Dataset asset already exists: ai-medical-chatbot-llama3@1.0.0
   path: azureml://subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourcegroups/rg-aml-yw-dos/workspaces/aml-ww-yw-dos/datastores/workspaceblobstore/paths/datasets/ai-medical-chatbot-llama3/v1.0.0/
   → skipping prep job; the training job below will mount this asset.


## 3. Submit the GPU fine-tuning job

The job mounts the dataset asset (`Input(type=URI_FOLDER, ...)`) into the
container; `fine_tune_llama_3_doctor.py` reads it via
`datasets.load_from_disk(--data_dir)`. No HF download happens on the GPU.


In [9]:
import datetime

# override the number of epochs for a quick smoke test; 
# the prep job above is the slow part, so we want to be able to 
# iterate quickly on the training job below without re-running prep.
NUM_EPOCHS    = 1

job_timestamp   = datetime.datetime.now().strftime("%Y%m%d.%H%M%S")
ft_display_name = f"ft-{FINETUNED_MODEL}-{job_timestamp}"

fine_tune_job = command(
    code="./src",
    command=(
        "python fine_tune_llama_3_doctor.py "
        "--data_dir=${{inputs.data}} "
        f"--base_model={BASE_MODEL} "
        f"--finetuned_model={FINETUNED_MODEL} "
        f"--num_epochs={NUM_EPOCHS} "
        f"--max_steps={MAX_STEPS} "
        f"--batch_size={BATCH_SIZE} "
        f"--grad_accum={GRAD_ACCUM} "
        f"--learning_rate={LEARNING_RATE} "
        f"--lora_r={LORA_R} "
        f"--lora_alpha={LORA_ALPHA}"
    ),
    inputs={
        "data": Input(
            type=AssetTypes.URI_FOLDER,
            path=f"azureml:{DATA_ASSET}:{DATA_VERSION}",
            mode="ro_mount",
        ),
    },
    environment=f"{ENV_NAME}@{ENV_VERSION}",
    compute=GPU_COMPUTE,
    identity=UserIdentityConfiguration(),
    experiment_name="llama3-sft",
    display_name=ft_display_name,
    description=(
        f"SFT {BASE_MODEL} on {DATA_ASSET}@{DATA_VERSION} "
        f"(epochs={NUM_EPOCHS}, max_steps={MAX_STEPS}, "
        f"batch={BATCH_SIZE}x{GRAD_ACCUM}, lr={LEARNING_RATE}, "
        f"LoRA r={LORA_R}/α={LORA_ALPHA})."
    ),
)

submitted = ml_client.jobs.create_or_update(fine_tune_job)
print(f"✅ Submitted training job: {submitted.name}")
print(f"   display    : {submitted.display_name}")
print(f"   status     : {submitted.status}")
print(f"   studio URL : {submitted.studio_url}")


Uploading src (0.02 MBs): 100%|██████████| 21010/21010 [00:00<00:00, 207179.45it/s]




✅ Submitted training job: orange_wing_vcdrd01436
   display    : ft-llama3-8b-chat-doctor-20260603.183415
   status     : Starting
   studio URL : https://ml.azure.com/runs/orange_wing_vcdrd01436?wsid=/subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourcegroups/rg-aml-yw-dos/workspaces/aml-ww-yw-dos&tid=787eb5ff-f2ee-4965-8074-edbba3402c84


## 4. Stream logs + download outputs

Blocks until the job is terminal. Falls back to polling if `stream` is
interrupted (e.g. SIGINT, network blip). Downloaded artifacts land under
`.job-logs/<job_name>/named-outputs/default/...`.


In [5]:
import time
from pathlib import Path

POLL_INTERVAL_S = 30
MAX_WAIT_S      = 4 * 3600          # 4 h ceiling

job_name = submitted.name
print(f"Streaming logs for '{job_name}' (blocks until terminal) ...")
try:
    ml_client.jobs.stream(job_name)
except Exception as e:
    print(f"⚠️  Stream interrupted: {e} — falling back to polling.")
    deadline = time.time() + MAX_WAIT_S
    terminal = {"Completed", "Failed", "Canceled", "NotResponding"}
    while time.time() < deadline:
        j = ml_client.jobs.get(job_name)
        print(f"  status={j.status}")
        if j.status in terminal:
            break
        time.sleep(POLL_INTERVAL_S)

j = ml_client.jobs.get(job_name)
print(f"\nFinal status : {j.status}")
print(f"Studio URL   : {j.studio_url}")

log_root = Path(".job-logs") / job_name
log_root.mkdir(parents=True, exist_ok=True)
print(f"\nDownloading outputs + logs → {log_root.resolve()} ...")
try:
    ml_client.jobs.download(name=job_name, download_path=str(log_root), all=True)
    print("✅ Downloaded.")
except Exception as e:
    print(f"⚠️  Download failed: {e}")

print("\nKey artifacts (if training succeeded):")
for sub in (
    f"named-outputs/default/{FINETUNED_MODEL}",
    f"named-outputs/default/{FINETUNED_MODEL}_config",
    f"named-outputs/default/{FINETUNED_MODEL}_full",
):
    p = log_root / sub
    if p.exists():
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f"  ✅ {sub}  ({n} files)")
    else:
        print(f"  ⚠️  {sub}  (missing)")

Streaming logs for 'green_coat_v6klnpkqvy' (blocks until terminal) ...
RunId: green_coat_v6klnpkqvy
Web View: https://ml.azure.com/runs/green_coat_v6klnpkqvy?wsid=/subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourcegroups/rg-aml-yw-dos/workspaces/aml-ww-yw-dos

Streaming user_logs/std_log.txt

W0603 13:33:47.619000 73 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0603 13:33:47.641000 73 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Arguments: ['fine_tune_llama_3_doctor.py', '--data_dir=/mnt/azureml/cr/j/d21891d8988949738531373210765388/cap/data-capability/wd/INPUT_data', '--

Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy 'https://amlwwywdos1036066399.blob.core.windows.net/azureml/ExperimentRun/dcid.green_coat_v6klnpkqvy' '.job-logs/green_coat_v6klnpkqvy/artifacts' 

See https://learn.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more information.
Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy 'https://amlwwywdos1036066399.blob.core.windows.net/azureml/ExperimentRun/dcid.green_coat_v6klnpkqvy' '.job-logs/green_coat_v6klnpkqvy/artifacts' 

See https://learn.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more information.
Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azco

✅ Downloaded.

Key artifacts (if training succeeded):
  ⚠️  named-outputs/default/llama3-8b-chat-doctor  (missing)
  ⚠️  named-outputs/default/llama3-8b-chat-doctor_config  (missing)
  ⚠️  named-outputs/default/llama3-8b-chat-doctor_full  (missing)


## Notes

### Why split prep from training
- GPU compute is ~10× the cost of CPU. Prep is network + tokenizer work,
  not GPU work — keep it on cheap hardware.
- The dataset asset is **pinned + reproducible** by version. Subsequent
  training runs skip the entire HF download + tokenization step.
- The training script becomes **offline-safe**: no HF round-trip at runtime,
  which also lets it run in a private-VNet workspace without an HF proxy.

### Bumping the dataset
Change `DATA_VERSION` whenever you change `PREP_NUM_ROWS`,
`PREP_TOKENIZER_MODEL`, `PREP_DATASET_NAME`, or the prep script. The blob
path embeds the version so old asset versions stay intact and re-runnable.

### Switching prep to a real CPU cluster
Set `PREP_COMPUTE = "cpu-cluster"` (or whatever you name it) and create a
cluster like `Standard_DS3_v2` with 0–1 nodes via `az ml compute create`.
That cluster also needs `Storage Blob Data Contributor` on the workspace
storage (re-run Step 3 of `00_aml_cc_prepare_sub_environment.ipynb` for the
new compute's MI principal).

### Inference / deployment
The merged full model lives under
`./outputs/<FINETUNED_MODEL>_full/` inside the job and is uploaded by AML.
After download, register as an AML Model asset and deploy to an online
endpoint (or pull into a downstream notebook for batch eval).
